## Import libraries

In [16]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.compose import ColumnTransformer

# Models
from sklearn.linear_model import LinearRegression, LassoCV, Ridge, ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor

# Metrics
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
    median_absolute_error,
    explained_variance_score,
    max_error
)

## Load the dataset

In [ ]:
url = "https://raw.githubusercontent.com/Abhishek-Janjal/regression/main/forest%2Bfires/forestfires.csv"
df = pd.read_csv(url)

df.head()

,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain,area
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,0.0
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,0.0
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0,0.0
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2,0.0
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0,0.0


In [10]:
import numpy as np

month_map = {'jan':1,'feb':2,'mar':3,'apr':4,'may':5,'jun':6,
             'jul':7,'aug':8,'sep':9,'oct':10,'nov':11,'dec':12}

day_map = {'mon':1,'tue':2,'wed':3,'thu':4,'fri':5,'sat':6,'sun':7}

df['month'] = df['month'].map(month_map)
df['day'] = df['day'].map(day_map)

# cyclic transform
df['month_sin'] = np.sin(2*np.pi*df['month']/12)
df['month_cos'] = np.cos(2*np.pi*df['month']/12)

df['day_sin'] = np.sin(2*np.pi*df['day']/7)
df['day_cos'] = np.cos(2*np.pi*df['day']/7)

X = df[['month_sin','month_cos','day_sin','day_cos',
        'FFMC','DMC','DC','ISI','temp','RH','wind','rain']]
y = np.log1p(df['area'])  # Log-transforming the target variable to handle skewness

## Train-test split

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Preprocessing

In [12]:
num_features = X.columns

trans = ColumnTransformer([
    ('num', StandardScaler(), num_features)
])

## Pipeline

In [13]:

pipe = Pipeline([
    ('transform', trans),
    ('scale', MinMaxScaler()),   # optional (can remove redundancy)
    ('clf', LinearRegression())
])

# Grid parameters for different models
grid_param = [

    # Decision Tree
    {
        "clf": [DecisionTreeRegressor()],
        "clf__splitter": ["best", "random"],
        "clf__max_depth": [5, 7, 10],
        "clf__min_samples_leaf": [2, 4],
    },

    # Lasso (L1)
    {
        "clf": [LassoCV(cv=5)]
    },

    # Ridge (L2)
    {
        "clf": [Ridge()],
        "clf__alpha": [0.1, 1.0, 10.0]
    },

    # ElasticNet (L1 + L2)
    {
        "clf": [ElasticNet()],
        "clf__alpha": [0.1, 1.0],
        "clf__l1_ratio": [0.3, 0.5, 0.7]
    },

    # Support Vector Regressor
    {
        "clf": [SVR()],
        "clf__kernel": ['linear', 'rbf', 'poly'],
        "clf__C": [0.1, 1, 10]
    },

    # AdaBoost
    {
        "clf": [AdaBoostRegressor()],
        "clf__n_estimators": [100, 200, 400],
        "clf__learning_rate": [0.01, 0.1],
        "clf__random_state": [1]
    },

    # Random Forest
    {
        "clf": [RandomForestRegressor()],
        "clf__n_estimators": [100, 200],
        "clf__max_depth": [10, 20],
        "clf__min_samples_split": [2, 10],
        "clf__min_samples_leaf": [1, 4],
        "clf__max_features": ['sqrt'],
        "clf__random_state": [1]
    }
]



## Training the model using GridSearchCV

In [14]:

# Grid SearchCV
grid = GridSearchCV(
    pipe,
    grid_param,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)


Fitting 5 folds for each of 53 candidates, totalling 265 fits


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('transform',
                                        ColumnTransformer(transformers=[('num',
                                                                         StandardScaler(),
                                                                         Index(['month_sin', 'month_cos', 'day_sin', 'day_cos', 'FFMC', 'DMC', 'DC',
       'ISI', 'temp', 'RH', 'wind', 'rain'],
      dtype='object'))])),
                                       ('scale', MinMaxScaler()),
                                       ('clf', LinearRegression())]),
             n_jobs=-1,
             param_grid=[{'clf': [DecisionTreeRegressor()],
                          'clf__...
                          'clf__kernel': ['linear', 'rbf', 'poly']},
                         {'clf': [AdaBoostRegressor()],
                          'clf__learning_rate': [0.01, 0.1],
                          'clf__n_estimators': [100, 200, 400],
                          'clf__random_state': [1]},
                         {'clf': [RandomForestRegressor()],
                          'clf__max_depth': [10, 20],
                          'clf__max_features': ['sqrt'],
                          'clf__min_samples_leaf': [1, 4],
                          'clf__min_samples_split': [2, 10],
                          'clf__n_estimators': [100, 200],
                          'clf__random_state': [1]}],
             scoring='r2', verbose=2)

## Evaluation and Results

In [17]:
# Results
print("Best Model:", grid.best_estimator_)
print("Best Params:", grid.best_params_)

# Evaluation
y_pred = grid.predict(X_test)

# Metrics
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
medae = median_absolute_error(y_test, y_pred)
evs = explained_variance_score(y_test, y_pred)
max_err = max_error(y_test, y_pred)

# Print nicely
print("📊 Regression Metrics")
print("----------------------")
print(f"R2 Score               : {r2:.4f}")
print(f"Adjusted R2            : {1 - (1-r2)*(len(y_test)-1)/(len(y_test)-X_test.shape[1]-1):.4f}")
print(f"MSE                    : {mse:.4f}")
print(f"RMSE                   : {rmse:.4f}")
print(f"MAE                    : {mae:.4f}")
print(f"Median Absolute Error  : {medae:.4f}")
print(f"Explained Variance     : {evs:.4f}")
print(f"Max Error              : {max_err:.4f}")

Best Model: Pipeline(steps=[('transform',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  Index(['month_sin', 'month_cos', 'day_sin', 'day_cos', 'FFMC', 'DMC', 'DC',
       'ISI', 'temp', 'RH', 'wind', 'rain'],
      dtype='object'))])),
                ('scale', MinMaxScaler()), ('clf', ElasticNet(alpha=0.1))])
Best Params: {'clf': ElasticNet(), 'clf__alpha': 0.1, 'clf__l1_ratio': 0.5}
📊 Regression Metrics
----------------------
R2 Score               : -0.0006
Adjusted R2            : -0.1326
MSE                    : 2.1992
RMSE                   : 1.4830
MAE                    : 1.2028
Median Absolute Error  : 1.1036
Explained Variance     : 0.0000
Max Error              : 5.8920
